# C6 · Terrain analysis: slope, aspect and hillshade for landslide studies

> **Time:** about 2.5 hours  
> **You need:** Parts A and B (especially A5 · Functions for slope in degrees and percent, and A8 · NumPy arrays), and Lessons C1 to C5, especially the UTM 45N grid with 30 m cells (C4) and masks, `np.select` and `NaN` (C5).  
> **By the end you can:**
> 1. compute **slope** in degrees and percent from a DEM with `np.gradient` and the real cell size in metres, and recognise the classic mistake of treating degrees as metres,
> 2. compute **aspect** (the compass direction a slope faces) and a **hillshade** from your own formula, and check them against xrspatial and Matplotlib,
> 3. measure **local relief** with moving windows (`scipy.ndimage`), sort slopes into classes for hazard mapping, and draw a shaded-relief map.

Water, rock strength, rainfall and earthquakes all matter for landslides, but one factor appears in every landslide study: **how steep the ground is**. Slope is computed from a DEM, and so are the other basic terrain maps: which way slopes face (aspect), how rugged the land is (relief), and the shaded relief that makes a map readable.

The DEM of Part C covers the Melamchi area of Sindhupalchok district, struck by the Melamchi flood in June 2021. This lesson turns it into the terrain maps that a geohazard study starts from. Lesson C7 then compares them with landslide points.

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA).* The full licence notice is in `data/README.md`.

**Counting lines:** lines that only set up or show a plot (`fig, ax = plt.subplots()`, `plt.show()`) do not count toward the Q1/Q2/Q3 line counts.

**About the numbers:** reprojection and slope can differ very slightly between library versions, so your decimals may differ a little from the text.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 27 of 32 · [Course home](../README.md)  
← Previous: [C5 · Raster algebra and reclassification: elevation zones of the Kathmandu Valley](C05_Raster_algebra_and_reclassification.ipynb) · Next: [C7 · Vector and raster together: landslides, districts and slope](C07_Vector_and_raster_together.ipynb) →

## 1. Slope from a gradient

In Lesson A5 you turned a rise and a run into a slope: **gradient = rise / run**, slope in percent = gradient × 100, and slope in degrees = `math.degrees(math.atan(gradient))`.

On a grid, the rise is measured between **neighbouring cells**, and the run is the distance between them: the **cell size in metres**. `np.gradient(z, dy, dx)` does this for every cell at once:

- it returns two grids: the change **per row** (`dz_drow`, north–south) and the change **per column** (`dz_dcol`, west–east), each divided by the spacing you give;
- for an inner cell it uses the two neighbours on either side: (next − previous) / (2 × cell size), a **central difference**. Edge cells use their one inner neighbour.

The steepest gradient combines both directions, like the two sides of a right-angled triangle (Pythagoras): `np.sqrt(dz_drow**2 + dz_dcol**2)`. Then `np.arctan` and `np.degrees` turn it into degrees, cell by cell.

The example uses the made-up mountain of A8 and C1, with 500 m cells.

In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
import rioxarray as rxr
import xarray as xr
from rasterio.enums import Resampling
from rasterio.transform import xy

DATA = Path("..") / "data"
OUT = Path("..") / "outputs"
OUT.mkdir(exist_ok=True)

elev_m = np.array([
    [4210, 4350, 4480, 4520, 4400, 4290],   # row 0 (north edge)
    [4380, 4620, 4890, 4950, 4700, 4460],   # row 1
    [4510, 4870, 5230, 5410, 5020, 4630],   # row 2
    [4470, 4790, 5150, 5290, 4980, 4590],   # row 3
    [4300, 4520, 4760, 4810, 4650, 4410],   # row 4 (south edge)
])
cell_size_m = 500   # made-up mountain from A8 and C1: 500 m cells

dz_drow, dz_dcol = np.gradient(elev_m, cell_size_m, cell_size_m)   # rise per metre, row and column directions
gradient = np.sqrt(dz_drow ** 2 + dz_dcol ** 2)
toy_slope_deg = np.degrees(np.arctan(gradient))
print(np.round(toy_slope_deg, 1))

In [ ]:
# Check one cell by hand: row 1, column 2
across = (elev_m[1, 3] - elev_m[1, 1]) / (2 * cell_size_m)   # west to east
down = (elev_m[2, 2] - elev_m[0, 2]) / (2 * cell_size_m)     # north to south (row 0 to row 2)
print(across, down)
print(round(math.degrees(math.atan(math.sqrt(across ** 2 + down ** 2))), 1))

The slope grid has the same shape as the elevations: one slope per cell. The cells of the made-up mountain slope between about 21.8° and 44°.

The hand check for row 1, column 2 uses the cells to its west and east (4,620 and 4,950 m) and to its north and south (4,480 and 5,230 m). The rise is 0.33 m per metre across and 0.75 m per metre down, which combine to a slope of about **39.3°**, the same as the grid at [1, 2]. `np.gradient` is only this arithmetic, done for every cell.

`np.arctan` and `np.degrees` are the NumPy versions of A5's `math.atan` and `math.degrees`: they work on whole arrays.

### Practice

**Q1 · one line:** Convert `gradient` into a slope in **percent** (A5) and print it rounded to 0 decimals.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(np.round(gradient * 100))
```

A gradient of 0.82 is 82%. Percent slope is common in road and site engineering; degrees are usual in landslide studies. 45° is 100%.
</details>

**Q2 · two lines:** Which cell of the made-up mountain is steepest? Use `argmax` and `np.unravel_index` (A8) on `toy_slope_deg`, and print the row, the column and the slope rounded to 1 decimal.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
row, col = np.unravel_index(toy_slope_deg.argmax(), toy_slope_deg.shape)
print(row, col, round(toy_slope_deg[row, col], 1))
```

Row 4, column 3, 44.0°: on the southern edge, where the ground falls from 5,290 m (row 3) to 4,810 m. The cell you checked by hand, row 1, column 2 (39.3°), is steep but not the steepest.
</details>

**Q3 · three lines:** What if the cells were 30 m instead of 500 m, with the same elevations? Recompute the slope with a spacing of 30 and print the mean slope of both versions, rounded to 1 decimal.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
dz_r30, dz_c30 = np.gradient(elev_m, 30, 30)
slope_30_deg = np.degrees(np.arctan(np.sqrt(dz_r30 ** 2 + dz_c30 ** 2)))
print(round(toy_slope_deg.mean(), 1), round(slope_30_deg.mean(), 1))
```

The same rises over a run about 17 times shorter: the mean slope jumps from about 34.7° to about 84.9°. The cell size you give `np.gradient` decides the answer, so it must be the true size in metres.
</details>

## 2. The slope of the DEM, in metres

Slope needs the **run in metres**. The DEM file is in degrees, so first put it on the UTM 45N grid with 30 m cells, exactly as in C4. Then the run between neighbours is 30 m in both directions.

`np.gradient` works on a NumPy array, so take `.values` from the DataArray. Cells next to the empty corners get `NaN`, because a difference with `NaN` is `NaN` (C5).

To save the slope as a GeoTIFF, write it with rasterio as in C1, giving the CRS and transform of `dem_utm`, `dtype="float32"` and `nodata=np.nan`.

**Predict:** the Kathmandu Valley floor is flat and the hills around it steep. What do you expect for the mean slope of the whole area: under 10°, about 20°, or over 40°?

In [ ]:
# As in C4: UTM 45N, 30 m cells, bilinear resampling; NaN where there is no data
dem = rxr.open_rasterio(DATA / "dem_kathmandu_melamchi.tif", masked=True).squeeze("band", drop=True)
dem_utm = dem.rio.reproject("EPSG:32645", resolution=30, resampling=Resampling.bilinear)
dem_m = dem_utm.values
cell_m = 30
CELL_KM2 = cell_m * cell_m / 1000000      # 0.0009 km² per cell

dz_drow, dz_dcol = np.gradient(dem_m, cell_m, cell_m)
slope_deg = np.degrees(np.arctan(np.sqrt(dz_drow ** 2 + dz_dcol ** 2)))

print(slope_deg.shape)
print("mean slope:", round(float(np.nanmean(slope_deg)), 1), "degrees")
print("steepest cell:", round(float(np.nanmax(slope_deg)), 1), "degrees")
print("area steeper than 30 degrees:", round((slope_deg > 30).sum() * CELL_KM2, 1), "km²")

In [ ]:
left, bottom, right, top = dem_utm.rio.bounds()
extent_km = [left / 1000, right / 1000, bottom / 1000, top / 1000]   # as in C4

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(slope_deg, extent=extent_km, cmap="YlOrRd", vmin=0, vmax=60)
fig.colorbar(im, ax=ax, label="Slope (degrees)", shrink=0.8)
ax.set_title("Slope of the DEM (UTM 45N, 30 m cells)")
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Northing (km)")
plt.show()

The mean slope is about **23.4°**, and the steepest cell is over 73°. About 887 km² is steeper than 30° (about 883 km² with older library versions): about 30% of the area with data.

On the map, pale yellow means gentle and dark red steep. The Kathmandu Valley floor, in the south-west, is almost white: slopes of a few degrees. The steep ground follows the river valleys of the north and east, where valley sides drop hundreds of metres. `vmax=60` keeps the colours readable: the few cells above 60° share the darkest red.

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA).*

### Practice

**Q1 · one line:** Print the mean slope in **percent**, rounded to 1 decimal. Use `np.tan(np.radians(slope_deg)) * 100` inside `np.nanmean`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round(float(np.nanmean(np.tan(np.radians(slope_deg)) * 100)), 1)
```

About 46%. It is not tan(23.4°) × 100 = 43%: the mean of percents is not the percent of the mean degrees, because tan curves upwards on steep ground.
</details>

**Q2 · two lines:** What share of the area with data is **gentle**, below 5°? Count those cells, divide by the cells with a slope value (`np.isfinite(slope_deg).sum()`, C5), and print the percentage rounded to 1 decimal.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
gentle = (slope_deg < 5).sum()
print(round(gentle / np.isfinite(slope_deg).sum() * 100, 1))
```

About 8.9%, mostly the valley floor and the wide river beds. `NaN < 5` is `False`, so the empty corners are not counted as gentle.
</details>

**Q3 · three lines:** Save the slope as `c6_slope_deg.tif` in `OUT` with rasterio, as in C1: one band, `dtype="float32"`, the CRS and transform of `dem_utm`, `nodata=np.nan`. Write `slope_deg.astype("float32")` into band 1.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
with rasterio.open(OUT / "c6_slope_deg.tif", "w", driver="GTiff", height=slope_deg.shape[0], width=slope_deg.shape[1], count=1,
                   dtype="float32", crs=dem_utm.rio.crs, transform=dem_utm.rio.transform(), nodata=np.nan) as dst:
    dst.write(slope_deg.astype("float32"), 1)
```

The file holds the slope with its decimals, on the same grid as `dem_utm`. Lesson C7 needs a slope grid; you can rebuild it there in a few lines, or read this file.
</details>

## 3. The classic mistake: degrees used as metres

The most common error in terrain analysis is computing slope on a grid in **degrees** as if its cells were metres. Nothing crashes. The numbers are simply wrong.

The DEM file's cells are 1/3600 of a degree. If you give `np.gradient` that spacing, the rise (in metres) is divided by a run of 0.000278 "units" instead of about 30 m: every slope comes out close to 90°. If you give no spacing at all, NumPy uses 1, as if the cells were 1 m apart.

There are two right ways:

1. reproject to a grid in metres first (step 2, C4), or
2. give the true cell size in metres, worked out from degrees as in C1: about 30.8 m north–south and 111,000 × cos(latitude) / 3600 m west–east.

The example shows the mistake **on purpose**, then the second fix.

In [ ]:
deg_m = dem.values                      # the DEM on its original grid in degrees
res_deg = abs(dem.rio.resolution()[0])  # 1/3600 of a degree

dr, dc = np.gradient(deg_m, res_deg, res_deg)         # WRONG: spacing in degrees
wrong_deg = np.degrees(np.arctan(np.sqrt(dr ** 2 + dc ** 2)))
print("spacing in degrees:", round(float(np.nanmean(wrong_deg)), 1))

dr, dc = np.gradient(deg_m)                           # WRONG: no spacing, so 1 m
no_spacing_deg = np.degrees(np.arctan(np.sqrt(dr ** 2 + dc ** 2)))
print("no spacing:", round(float(np.nanmean(no_spacing_deg)), 1))

mid_lat = float(dem.y.mean())
dy_m = res_deg * 111000                               # RIGHT: metres north-south
dx_m = res_deg * 111000 * math.cos(math.radians(mid_lat))   # and west-east (C1)
dr, dc = np.gradient(deg_m, dy_m, dx_m)
metres_deg = np.degrees(np.arctan(np.sqrt(dr ** 2 + dc ** 2)))
print("cell size in metres:", round(dy_m, 2), "x", round(dx_m, 2), "->", round(float(np.nanmean(metres_deg)), 1))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bins = np.arange(0, 91, 2)
ax.hist(wrong_deg.ravel(), bins=bins, color="crimson", alpha=0.6, label="spacing in degrees (wrong)")
ax.hist(slope_deg[np.isfinite(slope_deg)], bins=bins, color="steelblue", alpha=0.6, label="UTM grid, 30 m (right)")
ax.set_xlabel("Slope (degrees)")
ax.set_ylabel("Number of cells")
ax.set_title("The same DEM, two slope calculations")
ax.legend()
plt.show()

With the spacing in degrees, the mean slope is about **89.5°**: the whole landscape would be a cliff. With no spacing it is about 81.6°. Both are nonsense, and neither raised an error.

With the cell size in metres, about 30.83 m × 27.28 m, the mean is about **23.9°**, close to the 23.4° of the UTM grid. The small difference comes mostly from the bilinear resampling in C4, which smooths the surface a little. Notice that `dx_m` and `dy_m` differ: on a geographic grid, cells are not square on the ground (C1).

The histogram shows the warning sign. The wrong slopes pile up against 90°; real slopes in these hills spread from 0° to about 60°. **Always look at the range of your slopes.** A mean near 90° means a unit mistake, not a vertical landscape.

### Practice

**Q1 · one line:** What share of the wrong slopes (`wrong_deg`) is above 80°? Print the percentage, rounded to 1 decimal.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round(float((wrong_deg > 80).mean() * 100), 1)
```

Almost all of them, over 99%. On the right grid, cells above 80° do not exist at all. A quick share like this catches the mistake in seconds.
</details>

**Q2 · two lines:** Write a check that warns about the mistake: compute the mean of `wrong_deg`, then use `if` (A4) to print `"Check your units!"` when the mean slope is above 60°.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
mean_slope = float(np.nanmean(wrong_deg))
if mean_slope > 60: print("Check your units!")
```

It prints the warning. No real landscape has a mean slope above 60°; even the steep hills here average about 23°. A one-line sanity check like this belongs in every terrain script.
</details>

**Q3 · three lines:** How much do the cells change size from the south to the north edge of the DEM? Work out the west–east cell width in metres at the southern edge (27.55° N) and the northern edge (28.0° N), and print both rounded to 2 decimals.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
south_m = res_deg * 111000 * math.cos(math.radians(27.55))
north_m = res_deg * 111000 * math.cos(math.radians(28.0))
print(round(south_m, 2), round(north_m, 2))
```

About 27.34 m at the southern edge and 27.22 m at the northern edge. The difference is small here, but over a larger area one `dx_m` would no longer fit all rows. Reprojecting to UTM avoids that worry.
</details>

## 4. Aspect: which way a slope faces

**Aspect** is the compass direction a slope **faces**: the direction in which water would run straight downhill. It is measured clockwise from north: 0° north, 90° east, 180° south, 270° west, and 360° is north again. Flat ground has no aspect.

Aspect matters for hazards. Slopes facing south get more sun and dry faster; slopes facing the monsoon winds get more rain; and a slope that faces the same way as the rock layers dip is more likely to slide.

The downhill direction has an **east part**, `-dz_dcol` (ground falling towards the east), and a **north part**, `dz_drow` (rows run south, so ground rising southwards falls towards the north). `np.arctan2(east, north)` turns those two parts into a compass angle, from −180° to 180°. `% 360` (A2) folds the negative angles into 0° to 360°.

**Predict:** a slope that rises towards the north faces which way?

In [ ]:
def aspect_from(dz_drow, dz_dcol):
    """Compass direction (0-360 degrees, clockwise from north) that the slope faces."""
    return np.degrees(np.arctan2(-dz_dcol, dz_drow)) % 360

# Test on two made-up planes with 30 m cells
rises_north = np.array([[60.0] * 5, [45.0] * 5, [30.0] * 5, [15.0] * 5, [0.0] * 5])   # row 0 is north
rises_east = rises_north.T[:, ::-1]      # the same plane turned to rise towards the east
for name, plane in [("rises north", rises_north), ("rises east", rises_east)]:
    r, c = np.gradient(plane, 30, 30)
    print(name, "-> faces", aspect_from(r, c)[2, 2], "degrees")

In [ ]:
aspect_raw = aspect_from(dz_drow, dz_dcol)
aspect_deg = np.where(slope_deg < 1, np.nan, aspect_raw)   # flat ground (< 1 degree) has no aspect

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(aspect_deg, extent=extent_km, cmap="twilight", vmin=0, vmax=360)
cbar = fig.colorbar(im, ax=ax, shrink=0.8, ticks=[0, 90, 180, 270, 360], label="Aspect")
cbar.ax.set_yticklabels(["N (0°)", "E (90°)", "S (180°)", "W (270°)", "N (360°)"])
ax.set_title("Aspect: the direction each slope faces")
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Northing (km)")
plt.show()
print("flat cells:", int((slope_deg < 1).sum()))

The plane that rises towards the north **faces south**: 180°. The plane that rises towards the east faces west: 270°. Testing a formula on a surface whose answer you know is the best way to catch sign mistakes; here they would swap north and south.

`rises_north.T[:, ::-1]` turns the plane: `.T` swaps rows and columns, and `[:, ::-1]` reverses the order of the columns (A3 slicing), so the values grow towards the east.

The aspect map uses `"twilight"`, a **cyclic** colour map: its two ends have the same colour, because 0° and 360° are the same direction. Every ridge shows two colours meeting, its two sides facing opposite ways. Flat cells are blank: about 49,000 cells are below 1°, scattered over the valley floor and the wide river beds. The valley floor still looks speckled, because the Copernicus DEM measures roofs and trees (C1): a "flat" city has many small slopes.

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA).*

### Practice

**Q1 · one line:** What is the mean of `aspect_deg`? Use `np.nanmean`, round to 1 decimal, and think about whether the result means anything.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round(float(np.nanmean(aspect_deg)), 1)
```

About 181.7°, "south". But the mean of 350° and 10° (both nearly north) is 180°, south! Angles wrap round, so an ordinary mean of directions is meaningless. Count directions in sectors instead (review 2).
</details>

**Q2 · two lines:** What share of the sloping cells faces **south-ish**, between 135° and 225°? Divide the count by the number of cells with an aspect, `np.isfinite(aspect_deg).sum()`, and print the percentage rounded to 1 decimal.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
south = (aspect_deg >= 135) & (aspect_deg < 225)
print(round(south.sum() / np.isfinite(aspect_deg).sum() * 100, 1))
```

About 26.5%, a little more than a quarter. A landscape with no preferred direction would give 25%, one quarter of the compass.
</details>

**Q3 · three lines:** Test `aspect_from` on a third plane: one that rises towards the **south**. Build it by reversing the rows of `rises_north` with `[::-1]`, take its gradient with 30 m spacing, and print the aspect of the centre cell.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
rises_south = rises_north[::-1]
r, c = np.gradient(rises_south, 30, 30)
print(aspect_from(r, c)[2, 2])
```

0.0 degrees: a slope rising towards the south faces **north**, as it should. Three tests with known answers, three correct results: now the formula can be trusted on the DEM.
</details>

## 5. Hillshade: lighting the terrain

A **hillshade** shows the land as if lit by a low sun. Slopes facing the sun are bright, slopes facing away are dark. Our eyes read the result as relief.

The sun is placed with two angles: its **azimuth** (the compass direction it shines from) and its **altitude** (its angle above the horizon). Map makers put it in the **north-west (315°)** at **45°**, so that the light comes from the top-left of the map.

The brightness of a cell is the cosine of the angle between the sun and the ground's surface:

brightness = cos(zenith) × cos(slope) + sin(zenith) × sin(slope) × cos(azimuth − aspect),

where the **zenith** = 90° − altitude. Flat ground gets cos(zenith); a slope facing the sun gets more; a slope facing away gets less, and below 0 it lies in full shadow. `np.clip(values, 0, 1)` raises anything below 0 to 0.

In [ ]:
def hillshade_from(slope_deg, aspect_deg, azimuth_deg=315, altitude_deg=45):
    """Brightness from 0 (dark) to 1 (fully lit), for a sun at the given azimuth and altitude."""
    zenith = np.radians(90 - altitude_deg)
    slope = np.radians(slope_deg)
    shade = np.cos(zenith) * np.cos(slope) + np.sin(zenith) * np.sin(slope) * np.cos(np.radians(azimuth_deg - aspect_deg))
    return np.clip(shade, 0, 1)      # below 0 = facing away from the sun: full shadow

shade = hillshade_from(slope_deg, aspect_raw)                       # sun in the north-west
shade_se = hillshade_from(slope_deg, aspect_raw, azimuth_deg=135)   # sun in the south-east
print("flat ground:", round(float(hillshade_from(0, 0)), 3))
print("mean brightness:", round(float(np.nanmean(shade)), 3))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, grid, title in zip(axes, [shade, shade_se], ["Sun in the north-west (315°)", "Sun in the south-east (135°)"]):
    ax.imshow(grid, extent=extent_km, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel("Easting (km)")
axes[0].set_ylabel("Northing (km)")
plt.show()

In [ ]:
# Hillshade + elevation: a shaded-relief map with districts and headquarters (in UTM metres)
districts = gpd.read_file(DATA / "nepal_districts.geojson").to_crs(32645)
hq = gpd.read_file(DATA / "district_headquarters.geojson").to_crs(32645)
hq_in = hq.cx[left:right, bottom:top]          # .cx box selection, B5
extent_m = [left, right, bottom, top]

fig, ax = plt.subplots(figsize=(10, 8.5))
ax.imshow(shade, extent=extent_m, cmap="gray", vmin=0, vmax=1)
im = ax.imshow(dem_m, extent=extent_m, cmap="terrain", alpha=0.45)     # see-through colours on top
districts.boundary.plot(ax=ax, color="black", linewidth=0.8)
hq_in.plot(ax=ax, color="red", edgecolor="white", markersize=40)
for name, point in zip(hq_in["name"], hq_in.geometry):
    ax.annotate(name, (point.x, point.y), xytext=(5, 5), textcoords="offset points", fontsize=10)
ax.plot([left + 3000, left + 13000], [bottom + 3000, bottom + 3000], color="black", linewidth=4)  # 10 km bar
ax.text(left + 8000, bottom + 4200, "10 km", ha="center", va="bottom", fontsize=11)
ax.set_xlim(left, right)
ax.set_ylim(bottom, top)
ax.set_axis_off()
fig.colorbar(im, ax=ax, label="Elevation (m)", shrink=0.7)
ax.set_title("Kathmandu Valley and Melamchi area: shaded relief (sun from the north-west)")
plt.show()

Flat ground gets a brightness of 0.707, the cosine of a 45° zenith. The mean brightness of the whole area is about 0.634.

The two hillshades are made from the same slope and aspect; only the sun moved. With the sun in the north-west, ridges stand up and valleys sink in. With the sun in the south-east, many people see the relief **inverted**: valleys look like ridges. This illusion is why maps light the land from the top-left.

The shaded-relief map stacks layers, as in B11. The grey hillshade at the bottom gives the shape, the `terrain` colours at 45% opacity (`alpha=0.45`) give the elevation, and the district boundaries and headquarters, converted to UTM 45N (B4), give the places. Everything shares one CRS, so it lines up. The 10 km bar is a line 10,000 m long, drawn in map units; `set_axis_off` hides the coordinates, which a reader of this map does not need.

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA).*

### Practice

**Q1 · one line:** How bright is a 30° slope facing **straight at** the sun (aspect 315°)? Call `hillshade_from` with one slope and one aspect.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
hillshade_from(30, 315)
```

About 0.966: almost full light. The ground is tilted towards the sun, so the light hits it nearly head-on.
</details>

**Q2 · two lines:** What share of the area lies in **full shadow** (brightness 0) when the sun is low, at 15° altitude in the north-west? Print the percentage of cells with data, rounded to 1 decimal.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
low_sun = hillshade_from(slope_deg, aspect_raw, altitude_deg=15)
print(round((low_sun == 0).sum() / np.isfinite(low_sun).sum() * 100, 1))
```

About 23.4% of the cells. With a low sun, every slope steeper than the sun's altitude and facing away goes completely dark. A low sun exaggerates relief, which helps to spot small landforms such as old landslide scarps.
</details>

**Q3 · three lines:** Draw a hillshade with the sun **overhead** (altitude 90°). Plot it in grey with `extent=extent_km` and a title. What does it show?

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
overhead = hillshade_from(slope_deg, aspect_raw, altitude_deg=90)
fig, ax = plt.subplots(figsize=(8, 6.5))
ax.imshow(overhead, extent=extent_km, cmap="gray", vmin=0, vmax=1)
ax.set_title("Hillshade, sun overhead")
plt.show()
```

With the sun overhead, the zenith is 0, so brightness = cos(slope): the map is a picture of **slope** alone, with flat ground white and steep ground dark. Aspect no longer matters.
</details>

## 6. Checking your work against libraries

Libraries compute terrain too. **xrspatial** offers `slope`, `aspect` and `hillshade` for DataArrays. It takes the cell size from the x and y coordinates, so the DataArray must be in **metres**.

xrspatial uses **Horn's method**: it weighs all eight neighbours of a cell, not only the four that `np.gradient` uses. The results should be close to yours, but not identical.

Never trust a tool, yours or a library's, until it passes a test with a known answer. The second example tests three hillshades on a plane that rises 15 m per 30 m cell towards the north: slope 26.6°, aspect 180°. With the sun at 315° and 45°, the formula of step 5 gives cos 45° × cos 26.6° + sin 45° × sin 26.6° × cos 135° ≈ **0.41**.

In [ ]:
from xrspatial import slope, aspect, hillshade

lib_slope = slope(dem_utm).values
print("mean slope, yours:", round(float(np.nanmean(slope_deg)), 1), "| xrspatial:", round(float(np.nanmean(lib_slope)), 1))
print("mean absolute difference:", round(float(np.nanmean(np.abs(lib_slope - slope_deg))), 2), "degrees")

lib_aspect = aspect(dem_utm).values
lib_aspect = np.where(lib_aspect < 0, np.nan, lib_aspect)        # xrspatial marks flat cells with -1
turn = np.abs((lib_aspect - aspect_deg + 180) % 360 - 180)        # difference of two directions, 0 to 180
print("median aspect difference:", round(float(np.nanmedian(turn)), 2), "degrees")

print("xrspatial on the grid in DEGREES:", round(float(slope(dem).mean()), 1), "degrees")

In [ ]:
import xrspatial
from matplotlib.colors import LightSource

x_m = np.arange(5) * 30.0
y_m = 3000000 - np.arange(5) * 30.0      # northings fall row by row: north is up
plane = xr.DataArray(rises_north, coords={"y": y_m, "x": x_m}, dims=("y", "x"))

print("formula:", round(float(hillshade_from(26.565, 180)), 3))
print("Matplotlib LightSource:", round(float(LightSource(azdeg=315, altdeg=45).hillshade(rises_north, dx=30, dy=30)[2, 2]), 3))
print("xrspatial", xrspatial.__version__, ":", round(float(hillshade(plane, azimuth=315, angle_altitude=45)[2, 2]), 3))

xrspatial's mean slope is about 23.1° to 23.2° (depending on the version), yours about 23.4°, and cell by cell they differ by about 0.7° on average. Horn's method smooths a little over eight neighbours, so it gives slightly gentler slopes. Half of all aspect directions differ by less than about 1.7°. Your slope and aspect pass the comparison.

Given the grid in **degrees**, xrspatial reports a mean slope of about 89.8°: a library repeats the classic mistake of step 3 if you feed it degrees. Recent versions print a warning about it; older ones stay silent.

The plane test: your formula and Matplotlib's `LightSource` agree on **0.409**. xrspatial's `hillshade` gives a different number, and which number depends on its version. Version 0.10 lights the map as if the sun stood in the south-west, mirroring north and south; version 0.5 ignores the cell size and uses another brightness scale. Its slope and aspect are fine, but its hillshade fails the test, so this course uses your formula or `LightSource` (as C7 does).

### Practice

**Q1 · one line:** How many cells have no xrspatial slope? Count the `NaN` values in `lib_slope` with `np.isnan` and `.sum()`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
np.isnan(lib_slope).sum()
```

About 90,100 cells: the empty corners plus one ring of cells around them and around the edge of the grid, where a cell lacks some of its eight neighbours.
</details>

**Q2 · two lines:** Where do the two slope methods differ most? Find the largest absolute difference with `np.nanmax` and print it rounded to 1 decimal, next to the share of cells that differ by more than 2°.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
diff_deg = np.abs(lib_slope - slope_deg)
print(round(float(np.nanmax(diff_deg)), 1), round(float((diff_deg > 2).sum() / np.isfinite(diff_deg).sum() * 100), 1))
```

The largest difference is about 10° to 11° (depending on the version), but only about 5% of the cells differ by more than 2°. The big differences sit on sharp ridges and gully floors, where four and eight neighbours see different shapes.
</details>

**Q3 · three lines:** Test xrspatial's `slope` and `aspect` on the same `plane`. Print the slope and aspect of the centre cell `[2, 2]`, each wrapped in `float()`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
plane_slope = float(slope(plane)[2, 2])
plane_aspect = float(aspect(plane)[2, 2])
print(round(plane_slope, 1), plane_aspect)
```

26.6° and 180.0°: the right answers. xrspatial's slope and aspect pass the same test that its hillshade failed. Test each function separately; one good result says nothing about the others.
</details>

## 7. Moving windows: local relief and smoothing

A **moving window** (or **filter**, or **kernel**) slides over the grid. For every cell it looks at a small block around it, for example 5 × 5 cells, and writes one result back to the centre cell. Slope was already a tiny window operation.

`scipy.ndimage` has fast filters:

- `ndimage.maximum_filter(grid, size=5)` and `ndimage.minimum_filter(grid, size=5)`: the highest and lowest value in each 5 × 5 window. Their difference is the **local relief**, the height range inside 150 m × 150 m: a simple measure of how rugged the ground is.
- `ndimage.uniform_filter(grid, size=3)`: the **mean** of each 3 × 3 window, which **smooths** the surface. (An older notebook behind this course multiplied such a mean by 2 by mistake; `uniform_filter` gives the plain mean.)

These filters do not skip `NaN`. The example fills the empty cells with 0 **temporarily**, then blanks every cell whose window touched an empty cell.

In [ ]:
from scipy import ndimage

is_empty = np.isnan(dem_m)
filled_m = np.where(is_empty, 0, dem_m)                          # temporary fill
touches_empty = ndimage.maximum_filter(is_empty, size=5)         # True where the 5x5 window holds an empty cell

relief_m = ndimage.maximum_filter(filled_m, size=5) - ndimage.minimum_filter(filled_m, size=5)
relief_m = np.where(touches_empty, np.nan, relief_m)
print("local relief in 150 m x 150 m: mean", round(float(np.nanmean(relief_m)), 1), "m, largest", round(float(np.nanmax(relief_m)), 1), "m")

smooth_m = ndimage.uniform_filter(filled_m, size=3)
smooth_m = np.where(ndimage.maximum_filter(is_empty, size=3), np.nan, smooth_m)
dr, dc = np.gradient(smooth_m, cell_m, cell_m)
smooth_slope_deg = np.degrees(np.arctan(np.sqrt(dr ** 2 + dc ** 2)))
print("mean slope, original:", round(float(np.nanmean(slope_deg)), 1), "| smoothed:", round(float(np.nanmean(smooth_slope_deg)), 1))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(relief_m, extent=extent_km, cmap="magma_r", vmin=0, vmax=200)
fig.colorbar(im, ax=ax, label="Local relief in a 5 x 5 window (m)", shrink=0.8)
ax.set_title("Local relief: height range within 150 m")
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Northing (km)")
plt.show()

Within 150 m × 150 m, the ground rises and falls by about 68 m on average, and by up to about 420 m in the most rugged spot.

The 3 × 3 mean smooths the DEM, and the mean slope drops from about 23.4° to about 22.3°. Smoothing, like the coarser cells of C4, makes slopes gentler. Do it only on purpose, for example to remove noise, and say so.

`is_empty` is a True/False grid; `maximum_filter` on it is `True` wherever any cell of the window is `True`. That one line finds every window that touched the empty corners.

The relief map looks much like the slope map: rugged ground is steep ground. Dark cells line the steep valley sides in the north and east; the valley floor in the south-west and the wide river beds are pale.

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA).*

### Practice

**Q1 · one line:** What happens without the temporary fill? Apply `ndimage.uniform_filter` to `dem_m` itself (with its `NaN` corners) and count the `NaN` cells of the result.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
np.isnan(ndimage.uniform_filter(dem_m, size=3)).sum()
```

Every one of the 3,358,817 cells is `NaN`. This filter runs along whole rows and columns, so a single `NaN` spreads across the entire grid. Hence the fill-then-blank pattern of the example.
</details>

**Q2 · two lines:** Compute the local relief in a **3 × 3** window (90 m × 90 m) with the same pattern, and print its mean rounded to 1 decimal.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
relief_3_m = ndimage.maximum_filter(filled_m, size=3) - ndimage.minimum_filter(filled_m, size=3)
print(round(float(np.nanmean(np.where(ndimage.maximum_filter(is_empty, size=3), np.nan, relief_3_m))), 1))
```

About 35.2 m, about half the 5 × 5 value. Local relief depends on the window size, so always report the window with the number.
</details>

**Q3 · three lines:** Which share of the area has a 5 × 5 local relief above 100 m? Print the area in km² and the percentage of cells with a relief value, each rounded to 1 decimal.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
rugged = relief_m > 100
rugged_km2 = rugged.sum() * CELL_KM2
print(round(rugged_km2, 1), round(rugged.sum() / np.isfinite(relief_m).sum() * 100, 1))
```

About 490 km², roughly a sixth (about 16.8%) of the area, has more than 100 m of relief within 150 m. In 150 m, a 100 m rise means an average slope of more than 30°.
</details>

## 8. Slope classes for hazard mapping

Landslide studies often sort slopes into **classes**, because a map of four classes is easier to read and to compare than a map of every degree. Below are **teaching classes**, made up for this course and not an official susceptibility scheme:

| Class | Slope | Idea |
| --- | --- | --- |
| 1 · gentle | below 15° | landslides uncommon |
| 2 · moderate | 15° to 30° | landslides possible |
| 3 · steep | 30° to 45° | landslides common on weak ground |
| 4 · very steep | 45° and above | rock falls and shallow slides |

Real susceptibility maps combine slope with geology, land cover, rainfall and landslide records; slope alone never decides. Lesson C7 compares slope with landslide points.

The tools are those of C5: `np.select` for the classes, `np.unique` for the counts, cells × 0.0009 km² for the areas. The map drapes the classes over the hillshade, so the reader sees both steepness and the shape of the land.

In [ ]:
class_names = {1: "gentle (<15°)", 2: "moderate (15-30°)", 3: "steep (30-45°)", 4: "very steep (>=45°)"}
slope_class = np.select(
    [slope_deg < 15, (slope_deg >= 15) & (slope_deg < 30), (slope_deg >= 30) & (slope_deg < 45), slope_deg >= 45],
    [1, 2, 3, 4], default=0)                     # 0 = no data

codes, counts = np.unique(slope_class[slope_class > 0], return_counts=True)
class_table = pd.DataFrame({"class": codes, "name": [class_names[c] for c in codes], "cells": counts})
class_table["area_km2"] = class_table["cells"] * CELL_KM2
class_table["percent"] = class_table["cells"] / class_table["cells"].sum() * 100
class_table.round(1)

In [ ]:
class_map = np.where(slope_class == 0, np.nan, slope_class)

fig, ax = plt.subplots(figsize=(9, 7))
ax.imshow(shade, extent=extent_km, cmap="gray", vmin=0, vmax=1)
im = ax.imshow(class_map, extent=extent_km, cmap=plt.get_cmap("YlOrRd", 4), vmin=0.5, vmax=4.5, alpha=0.6)
cbar = fig.colorbar(im, ax=ax, shrink=0.8, ticks=[1, 2, 3, 4])
cbar.ax.set_yticklabels([class_names[c] for c in [1, 2, 3, 4]])
ax.set_title("Slope classes (teaching classes) over the hillshade")
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Northing (km)")
plt.show()

The moderate class is the largest, about 1,355 km² or 46% of the area. Gentle ground covers about 700 km² (24%), steep ground about 810 km² (28%) and very steep ground about 75 km² (2.6%). Together, steep and very steep cover the same area as the "steeper than 30°" count of step 2.

On the map, the steep and very steep classes cover most of the hills in the north, east and south-west, while the Kathmandu Valley floor and the wide river beds stay pale. The hillshade underneath shows which side of each valley is steep. This map is the starting point of a susceptibility study, not its result.

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA).*

### Practice

**Q1 · one line:** Look up the name of class 3 in `class_names`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
class_names[3]
```

'steep (30-45°)'. One dictionary feeds the table, the colour bar and any later report, so the names never disagree.
</details>

**Q2 · two lines:** How much of the area is steeper than 30°? Filter `class_table` for classes 3 and 4 and sum `area_km2`, rounded to 1 decimal.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
steep_classes = class_table[class_table["class"] >= 3]
print(round(steep_classes["area_km2"].sum(), 1))
```

About 887 km² (883 km² with older library versions), the same as the mask in step 2. Lesson C7 uses the same 30° threshold to compare slope with landslide points.
</details>

**Q3 · three lines:** Do coarse cells hide steep ground? Reproject `dem` to 90 m (`Resampling.average`, C4), compute its slope with a spacing of 90 m, and print the share of its cells steeper than 30°, next to the share on the 30 m grid. Round both to 1 decimal.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
dem_90 = dem.rio.reproject("EPSG:32645", resolution=90, resampling=Resampling.average).values
r90, c90 = np.gradient(dem_90, 90, 90)
slope_90 = np.degrees(np.arctan(np.sqrt(r90 ** 2 + c90 ** 2)))
print(round((slope_90 > 30).sum() / np.isfinite(slope_90).sum() * 100, 1), round((slope_deg > 30).sum() / np.isfinite(slope_deg).sum() * 100, 1))
```

About 20.0% of the 90 m cells are steeper than 30°, against about 30.1% of the 30 m cells: a third of the steep ground disappears. Coarse cells average away short steep slopes, so a hazard map built on them underestimates the steep ground. Use the finest DEM you can trust.
</details>

## 9. When it breaks: `math.atan` on a whole grid

In Lesson A5 you computed slope with `math.atan` and `math.degrees`. You try the same on the gradient grid. Run the cell.

In [ ]:
steepness = math.degrees(math.atan(gradient))

The last line of the error says:

`TypeError: only 0-dimensional arrays can be converted to Python scalars`

(Some NumPy versions word it slightly differently; the meaning is the same.)

- **TypeError**: a function got a value of the wrong type.
- **0-dimensional arrays** are arrays holding one single number. **Python scalars** are plain numbers such as `0.82`.
- The `math` module works on **one number at a time**. `gradient` is a whole grid, so `math.atan` cannot take it.

The fix: use NumPy's versions, which work cell by cell on whole arrays: `np.degrees(np.arctan(gradient))`. The same holds for `math.sqrt`, `math.cos` and `math.radians`: inside grid calculations, use `np.sqrt`, `np.cos` and `np.radians`.

**Try it:** fix the cell.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · A slope function you can trust

Write a function `slope_degrees(z, cell_m)` (A5) that returns the slope in degrees of a grid `z` with square cells of `cell_m` metres. Test it on a made-up plane that rises 30 m per 30 m cell (the centre should be 45°). Then use it to compare the mean slope of the 30 m grid with that of the 90 m grid from step 8, Q3.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
def slope_degrees(z, cell_m):
    """Slope in degrees of grid z with square cells of cell_m metres."""
    dz_r, dz_c = np.gradient(z, cell_m, cell_m)
    return np.degrees(np.arctan(np.sqrt(dz_r ** 2 + dz_c ** 2)))

test_plane = np.array([[120.0] * 5, [90.0] * 5, [60.0] * 5, [30.0] * 5, [0.0] * 5])
print("test:", slope_degrees(test_plane, 30)[2, 2])

grid_90 = dem.rio.reproject("EPSG:32645", resolution=90, resampling=Resampling.average).values
print("30 m:", round(float(np.nanmean(slope_degrees(dem_m, 30))), 1))
print("90 m:", round(float(np.nanmean(slope_degrees(grid_90, 90))), 1))
```

The test plane gives exactly 45.0°, so the function is right. The mean slope is about 23.4° on the 30 m grid but only about 20.8° on the 90 m grid: coarse cells make the whole landscape look gentler. A tested function is worth more than a fast one.
</details>

### Review 2 · Which way do the steep slopes face?

Sort every cell steeper than 30° into one of eight compass sectors: N, NE, E, SE, S, SW, W, NW. The sector number is `((aspect + 22.5) // 45) % 8`, which gives 0 for N (337.5° to 22.5°), 1 for NE, and so on. Count the cells per sector with pandas and print the percentages, rounded to 1 decimal.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
steep = slope_deg > 30
sector = ((aspect_deg[steep] + 22.5) // 45) % 8
names = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]

counts = pd.Series(sector.astype(int)).map(dict(enumerate(names))).value_counts()
percent = (counts / counts.sum() * 100).round(1)
print(percent.reindex(names))
```

Each sector holds roughly an eighth (12.5%) of the steep cells: south (about 15.4%) and south-west (about 14.0%) a little more, north-west (about 10.5%) a little less. No direction dominates strongly; explaining the small differences would need geology and climate data. `+ 22.5` shifts the sectors so that north runs from 337.5° to 22.5°; `// 45` numbers the 45° sectors; `% 8` folds 360° back to 0. `dict(enumerate(names))` maps 0 to "N", 1 to "NE", and so on, and `reindex` puts the sectors in compass order.
</details>

### Review 3 · An evening hillshade, saved for a report

Make a hillshade with an evening sun from the west (azimuth 270°) at 20° altitude. Plot it in grey with `extent=extent_km`, axis labels in km, and the Copernicus credit as a small caption with `fig.text`. Save the figure as `c6_evening_hillshade.png` in `OUT` at 300 dpi (A10) and print the share of cells in full shadow.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
evening = hillshade_from(slope_deg, aspect_raw, azimuth_deg=270, altitude_deg=20)

fig, ax = plt.subplots(figsize=(8, 6.5))
ax.imshow(evening, extent=extent_km, cmap="gray", vmin=0, vmax=1)
ax.set_title("Hillshade, evening sun from the west (270°, 20° altitude)")
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Northing (km)")
fig.text(0.01, 0.01, "Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018,\nprovided under COPERNICUS by the European Union and ESA)", fontsize=6)
fig.savefig(OUT / "c6_evening_hillshade.png", dpi=300, bbox_inches="tight")
plt.show()
print(round((evening == 0).sum() / np.isfinite(evening).sum() * 100, 1), "% in full shadow")
```

The east-facing slopes go dark and the west-facing ones glow. About 15.9% of the cells lie in full shadow. `fig.text` places text in figure coordinates, from (0, 0) at the bottom-left to (1, 1) at the top-right, so the credit travels with the saved image.
</details>

### Review 4 · The most rugged spot

Find the cell with the largest 5 × 5 local relief (`relief_m`, step 7). Turn its row and column into UTM coordinates with `xy` (C1) and `dem_utm.rio.transform()`, then into longitude and latitude with GeoPandas (B4). Print the relief, the slope of that cell and its longitude and latitude, rounded to 4 decimals.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
row, col = np.unravel_index(np.nanargmax(relief_m), relief_m.shape)
x_m, y_m = xy(dem_utm.rio.transform(), row, col)

spot = gpd.GeoSeries(gpd.points_from_xy([x_m], [y_m]), crs=32645).to_crs(4326).iloc[0]
print("relief:", round(float(relief_m[row, col]), 1), "m | slope:", round(float(slope_deg[row, col]), 1), "degrees")
print("lon:", round(spot.x, 4), "| lat:", round(spot.y, 4))
```

The most rugged 150 m × 150 m block rises about 420 m from its lowest to its highest cell. Its centre lies near the northern edge of the DEM, in the high ground of the north-east. `np.nanargmax` skips `NaN`, and the round trip from row and column to UTM to degrees uses C1's transform and B4's `to_crs`.
</details>

## Recap

- **Slope** = arctan of the gradient: `np.gradient(z, cell_m, cell_m)`, then `np.degrees(np.arctan(np.sqrt(dz_drow**2 + dz_dcol**2)))`. Percent = tan(slope) × 100.
- The cell size **must be in metres**: reproject to UTM 45N first (C4) or give the true metre spacing. Degrees used as metres give slopes near 90° with no error; always check the range.
- **Aspect** is the direction a slope faces, clockwise from north: `np.degrees(np.arctan2(-dz_dcol, dz_drow)) % 360`. Flat cells have none, and directions cannot be averaged like numbers.
- A **hillshade** lights the terrain from a sun at an azimuth and altitude (map convention 315°, 45°). Test any tool on a surface with a known answer: your formula and Matplotlib's `LightSource` pass; xrspatial's hillshade does not, while its slope and aspect do.
- **Moving windows** (`scipy.ndimage` maximum, minimum and uniform filters) give local relief and smoothing; fill `NaN` temporarily and blank the windows that touched it. **Slope classes** (teaching classes) and draped maps turn it all into a hazard picture.

### Words you learned

| Word | Meaning |
| --- | --- |
| slope | How steep the ground is: the angle (degrees) or rise/run × 100 (percent) |
| gradient | Rise per metre of run; tan(slope) |
| `np.gradient` | Change between neighbouring cells, divided by the spacing, for every cell |
| central difference | (next − previous) / (2 × spacing): the change across a cell from its two neighbours |
| aspect | The compass direction a slope faces, 0–360° clockwise from north |
| `np.arctan2` | The angle of a direction from its two parts; here (east, north) gives a compass bearing |
| cyclic colour map | A colour map whose two ends match, for angles such as aspect |
| hillshade | A grey image of the terrain lit by a sun at a chosen azimuth and altitude |
| azimuth / altitude | The compass direction of the sun / its angle above the horizon |
| zenith | 90° − altitude: the sun's angle from straight overhead |
| relief inversion | The illusion that valleys look like ridges when light comes from the bottom of the map |
| Horn's method | A slope formula that weighs all eight neighbours of a cell |
| moving window / filter / kernel | A small block that slides over a grid and computes one value per cell |
| local relief | Highest minus lowest elevation inside a window |
| smoothing | Replacing each cell with an average of its neighbours |
| slope classes | Ranges of slope used to simplify maps, here teaching classes |
| susceptibility | How prone ground is to landslides, judged from several factors |

## Next

**C7 · Vector and raster together:** read elevation and slope at the practice landslide points, compare them with the whole landscape, and summarise slope per district.

<!-- pygile-nav -->
**PyGILE Course** · lesson 27 of 32 · [Course home](../README.md)  
← Previous: [C5 · Raster algebra and reclassification: elevation zones of the Kathmandu Valley](C05_Raster_algebra_and_reclassification.ipynb) · Next: [C7 · Vector and raster together: landslides, districts and slope](C07_Vector_and_raster_together.ipynb) →